# Imports and SparkSession

In [1]:
import os

hf_cache = os.path.abspath("hf_cache")
os.makedirs(hf_cache, exist_ok=True)

os.environ["HF_HOME"] = hf_cache
os.environ["HF_TOKEN_PATH"] = os.path.join(hf_cache, "token")

In [2]:
# HuggingFace
from transformers import AutoTokenizer

# Spark functions
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, udf, concat
from pyspark.sql.types import ArrayType, IntegerType
from pyspark.ml.functions import array_to_vector
from pyspark.ml.feature import StringIndexer, MinMaxScaler
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [3]:
# spark session builder
spark = SparkSession.builder \
    .config("spark.driver.memory", "24g") \
    .config("spark.executor.memory", "24g") \
    .config("spark.executor.instances", 4) \
    .config("spark.driver.maxResultSize", "8g") \
    .getOrCreate()

In [4]:
spark

# Preprocessing
## Filtering Dataset

In [ ]:
# load dataset
df = spark.read.csv('shared/', header=True)
df.describe()

DataFrame[summary: string, title: string, post_id: string, over_18: string, subreddit: string, link_flair_text: string, self_text: string]

In [ ]:
filtered_df = df.filter(col("over_18")==False)\
                .filter(col("self_text").isNotNull() & 
                        (col("self_text") != "") & 
                        (col("self_text") != "[removed]") & 
                        (col("self_text") != "[deleted]"))

filtered_count = filtered_df.count()  # -> 97950710
print("The filtered data size is " + str( filtered_count /654221435*100) + "% of the original")# 21%

The filtered data size is 14.972103443843904% of the original


In [ ]:
# write filtered df to parquet file
filtered_df.write.parquet('shared_notebook/data/filtered', mode='overwrite')

In [ ]:
# reload dataset
df = spark.read.parquet("shared_notebook/data/filtered")

df.printSchema()

root
 |-- title: string (nullable = true)
 |-- post_id: string (nullable = true)
 |-- over_18: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- link_flair_text: string (nullable = true)
 |-- self_text: string (nullable = true)



In [ ]:
# sample first
df = df.sample(False, 0.01, seed=42)

print("Sample rows:", df.count())

Sample rows: 979208


In [9]:
# keep subreddits with enough examples
subreddit_counts = df.groupBy("subreddit").count()

valid_subreddits = (
    subreddit_counts
    .filter(col("count") >= 5)
    .select("subreddit")
)

In [10]:
# filter to valid subreddits
model_df = df.join(
    valid_subreddits,
    on="subreddit",
    how="inner"
)

print("Model rows:", model_df.count())
print("Number of subreddits:", model_df.select("subreddit").distinct().count())

Model rows: 848361
Number of subreddits: 21991


## NLP Tokenization and Encoding

In [11]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased",
    cache_dir=hf_cache,
    token=False
)

In [12]:
# turn text into token ids
def tokenize_text(text):
    if text is None:
        return []

    return tokenizer.encode(
        str(text),
        truncation=True,
        padding="max_length",
        max_length=32
    )

In [13]:
tokenize_udf = udf(
    tokenize_text,
    ArrayType(IntegerType())
)

In [14]:
# tiny tokenization test
tiny_df = df.limit(5)

tiny_tokenized = (
    df
    .withColumn("title_tokens", tokenize_udf(col("title")))
    .withColumn("self_text_tokens", tokenize_udf(col("self_text")))
)

tiny_tokenized.select(
    "subreddit",
    "title_tokens",
    "self_text_tokens"
).show(truncate=80)

+------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|   subreddit|                                                                    title_tokens|                                                                self_text_tokens|
+------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|brockhampton|[101, 2052, 1057, 2738, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...|[101, 3193, 8554, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...|
|       gopro|[101, 3087, 2182, 2145, 3594, 1037, 2175, 4013, 5394, 1022, 1029, 102, 0, 0, ...|[101, 10047, 5782, 2000, 2131, 1037, 2117, 2192, 2028, 2349, 2000, 5166, 1998...|
|slaythespire|[101, 2079, 1045, 2031, 2438, 2751, 2664, 1029, 102, 0, 0, 0, 0, 0, 0, 0, 0, ...|[101, 19827, 2751, 2

In [15]:
# tokenize modeling data
tokenized_df = (
    model_df
    .withColumn("title_tokens", tokenize_udf(col("title")))
    .withColumn("self_text_tokens", tokenize_udf(col("self_text")))
)

In [16]:
# combine title and self_text tokens
feature_df = tokenized_df.withColumn(
    "combined_tokens",
    concat(
        col("title_tokens"),
        col("self_text_tokens")
    )
)

In [17]:
# convert array<int> to Spark ML vector
feature_df = feature_df.withColumn(
    "raw_features",
    array_to_vector(col("combined_tokens"))
)

In [18]:
# scale features
scaler = MinMaxScaler(
    inputCol="raw_features",
    outputCol="features"
)

scaler_model = scaler.fit(feature_df)
feature_df = scaler_model.transform(feature_df)

In [19]:
# encode subreddit labels
label_indexer = StringIndexer(
    inputCol="subreddit",
    outputCol="label",
    handleInvalid="skip"
)

label_model = label_indexer.fit(feature_df)
feature_df = label_model.transform(feature_df)

# Model Training

In [20]:
# split into train, validation, and test
train_df, val_df, test_df = feature_df.randomSplit(
    [0.7, 0.15, 0.15],
    seed=42
)

print("Train:", train_df.count())
print("Validation:", val_df.count())
print("Test:", test_df.count())

Train: 593542
Validation: 127226
Test: 127593


In [21]:
# evaluation metrics
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

## First Pass 

In [ ]:
# decision tree model
dt = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    maxBins=32,
    impurity="gini",
    seed=42
)

dt_model = dt.fit(train_df)

In [ ]:
# predictions
dt_train_predictions = dt_model.transform(train_df)
dt_val_predictions = dt_model.transform(val_df)
dt_test_predictions = dt_model.transform(test_df)

In [ ]:
print("Decision Tree Results")

print("Train Accuracy:",
      accuracy_eval.evaluate(dt_train_predictions))

print("Validation Accuracy:",
      accuracy_eval.evaluate(dt_val_predictions))

print("Test Accuracy:",
      accuracy_eval.evaluate(dt_test_predictions))

print("Train F1:",
      f1_eval.evaluate(dt_train_predictions))

print("Validation F1:",
      f1_eval.evaluate(dt_val_predictions))

print("Test F1:",
      f1_eval.evaluate(dt_test_predictions))

Decision Tree Results
Train Accuracy: 0.0368971361757045
Validation Accuracy: 0.03660415323911779
Test Accuracy: 0.03651454233382709
Train F1: 0.015688694359323567
Validation F1: 0.015478156387307823
Test F1: 0.015124164847254387


In [ ]:
# random forest model
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=10,
    maxDepth=5,
    maxBins=32,
    seed=42
)

rf_model = rf.fit(train_df)

In [ ]:
# predictions
rf_train_predictions = rf_model.transform(train_df)
rf_val_predictions = rf_model.transform(val_df)
rf_test_predictions = rf_model.transform(test_df)

In [ ]:
print("Random Forest Results")

print("Train Accuracy:",
      accuracy_eval.evaluate(rf_train_predictions))

print("Validation Accuracy:",
      accuracy_eval.evaluate(rf_val_predictions))

print("Test Accuracy:",
      accuracy_eval.evaluate(rf_test_predictions))

print("Train F1:",
      f1_eval.evaluate(rf_train_predictions))

print("Validation F1:",
      f1_eval.evaluate(rf_val_predictions))

print("Test F1:",
      f1_eval.evaluate(rf_test_predictions))

Random Forest Results
Train Accuracy: 0.041136094834063976
Validation Accuracy: 0.04126515020514675
Test Accuracy: 0.04091133526133879
Train F1: 0.0133112475353809
Validation F1: 0.01339377043401687
Test F1: 0.013016102770163454


## Tuning and Results

In [28]:
# decision tree tuned
dt_tuned = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=3,
    maxBins=16,
    impurity="entropy",
    seed=42
)

dt_tuned_model = dt_tuned.fit(train_df)

In [29]:
# predictions
dt_tuned_train_predictions = dt_tuned_model.transform(train_df)
dt_tuned_val_predictions = dt_tuned_model.transform(val_df)
dt_tuned_test_predictions = dt_tuned_model.transform(test_df)

In [30]:
print("Tuned Decision Tree Results")

print("Train Accuracy:",
      accuracy_eval.evaluate(dt_tuned_train_predictions))

print("Validation Accuracy:",
      accuracy_eval.evaluate(dt_tuned_val_predictions))

print("Test Accuracy:",
      accuracy_eval.evaluate(dt_tuned_test_predictions))

print("Train F1:",
      f1_eval.evaluate(dt_tuned_train_predictions))

print("Validation F1:",
      f1_eval.evaluate(dt_tuned_val_predictions))

print("Test F1:",
      f1_eval.evaluate(dt_tuned_test_predictions))

Tuned Decision Tree Results
Train Accuracy: 0.02085446354259682
Validation Accuracy: 0.02074261550312043
Test Accuracy: 0.021631280712891773
Train F1: 0.0022149410792277656
Validation F1: 0.0021768773537772187
Test F1: 0.002286882373451695


I compared a baseline Decision Tree model, a tuned Decision Tree model, and a Random Forest model.

The baseline Decision Tree used:

maxDepth = 5
maxBins = 32
impurity = "gini"

The tuned Decision Tree used:

maxDepth = 3
maxBins = 16
impurity = "entropy"

The tuned Decision Tree performed worse than the baseline Decision Tree. Its test accuracy dropped from about 3.65% to 2.16%, and its F1 score also became much lower. I believe this happened because lowering the depth and bins made the model too simple, which caused more underfitting.

I also trained a Random Forest model, which performed slightly better than the Decision Tree. The Random Forest had a test accuracy of about 4.09%, which was the best among the models tested. However, the F1 score was still low, showing that overall prediction quality is still limited.

In [31]:
# random forest tuned
rf_tuned = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=15,
    maxDepth=5,
    maxBins=32,
    seed=42
)

rf_tuned_model = rf_tuned.fit(train_df)

In [32]:
# predictions
rf_tuned_train_predictions = rf_tuned_model.transform(train_df)
rf_tuned_val_predictions = rf_tuned_model.transform(val_df)
rf_tuned_test_predictions = rf_tuned_model.transform(test_df)

In [33]:
print("Tuned Random Forest Results")

print("Train Accuracy:",
      accuracy_eval.evaluate(rf_tuned_train_predictions))

print("Validation Accuracy:",
      accuracy_eval.evaluate(rf_tuned_val_predictions))

print("Test Accuracy:",
      accuracy_eval.evaluate(rf_tuned_test_predictions))

print("Train F1:",
      f1_eval.evaluate(rf_tuned_train_predictions))

print("Validation F1:",
      f1_eval.evaluate(rf_tuned_val_predictions))

print("Test F1:",
      f1_eval.evaluate(rf_tuned_test_predictions))

Tuned Random Forest Results
Train Accuracy: 0.04138376054264062
Validation Accuracy: 0.04122585006209423
Test Accuracy: 0.04125618176545735
Train F1: 0.013528217026358934
Validation F1: 0.013460583433274127
Test F1: 0.013302595437743054
